# EVALUATE ALL TRAINED MODELS
## Evaluate every checkpoint on train/valid/test splits and print a full comparison table.

**Prerequisites:** Mount Drive → Clone repo → Download data (cells 1-4)
**Then run:** cell 5 (main evaluation) + cell 6 (results)

In [ ]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except ImportError:
    print('Not in Colab — mount step skipped.')

In [ ]:
# ============================================================
# CELL 2 — CONFIG
# ============================================================

import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/Kandesfx/Training-Multimodal-Emotion-Analysis.git'
REPO_DIR = Path('/content/BCDA')

# GCS config (matching training notebooks)
USE_GCS = True
GCS_BUCKET = 'mer-data-bucket-kandesfx'

DRIVE_ROOT = Path('/content/drive/MyDrive/BCDA')

In [ ]:
# ============================================================
# CELL 3 — CLONE / PULL REPO
# ============================================================

import subprocess

if REPO_DIR.exists():
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=False)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f'Repo ready: {REPO_DIR}')

In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD DATA FROM GCS
# ============================================================

import os

os.makedirs('/content/data/MSA-Dataset', exist_ok=True)
os.makedirs('/content/checkpoints/phase1', exist_ok=True)

DATA_FILES = [
    'data/MSA-Dataset/aligned_50.pkl',
    'data/MSA-Dataset/unaligned_50.pkl',
]

CKPT_FILES = [
    'checkpoints/phase1/best_model_mult.pt',
    'checkpoints/phase1/best_model_mult_unaligned.pt',
    'checkpoints/phase1/best_model_improved_lstm.pt',
    'checkpoints/phase1/best_model_mult_emotion.pt',
    'checkpoints/phase1/best_model_mult_emotion_p1_focal.pt',
]

all_files = DATA_FILES + CKPT_FILES

if USE_GCS:
    from google.colab import auth
    print('Authenticating for GCS...')
    auth.authenticate_user()
    for f in all_files:
        src = f'gs://{GCS_BUCKET}/{f}'
        dst = f'/content/{f}'
        print(f'  Downloading: {f}')
        os.system(f'gsutil cp {src} {dst}')
else:
    print('USE_GCS=False — skipping GCS download.')
    print('Make sure data and checkpoints exist at:')
    print('  /content/data/MSA-Dataset/aligned_50.pkl')
    print('  /content/checkpoints/phase1/')

In [ ]:
# ============================================================
# CELL 5 — MAIN EVALUATION
# ============================================================

import warnings, json
import numpy as np
import pandas as pd
import torch
from torch import nn

from training.config_phase1 import Phase1Config, config as default_config
from training.dataset_mosei import create_dataloaders
from training.evaluator import compute_metrics
from training.evaluator_emotion import compute_emotion_metrics

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")


# ============================================================
# Model factory
# ============================================================
def build_model(cfg: Phase1Config, task_type: str):
    if cfg.model_type == 'early_fusion':
        from training.models.early_fusion import EarlyFusionLSTMRegressor
        return EarlyFusionLSTMRegressor(cfg.model)
    elif cfg.model_type == 'improved_lstm':
        from training.models.improved_lstm import ImprovedLSTMRegressor
        return ImprovedLSTMRegressor(cfg.model)
    elif cfg.model_type == 'mult':
        from training.models.mult import MulTRegressor
        cfg.mult_model.output_dim = 6 if task_type == 'emotion' else 1
        return MulTRegressor(cfg.mult_model)
    raise ValueError(f'Unsupported: {cfg.model_type}')


# ============================================================
# Evaluate one checkpoint on all splits
# ============================================================
def evaluate_checkpoint(ckpt_path: str, task_type: str, model_type: str,
                          pkl_path: str, label: str = None) -> dict:
    label = label or ckpt_path
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"  Task: {task_type}  |  Model: {model_type}")
    print('='*60)

    # Build config
    cfg = Phase1Config()
    cfg.model_type = model_type
    cfg.training.task_type = task_type
    cfg.training.batch_size = 32
    cfg.training.num_workers = 2
    cfg.apply_profile('colab')
    cfg.setup()

    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f"  Epoch: {ckpt.get('epoch', 'N/A')}  |  "
          f"Best metric: {ckpt.get('best_metric', 'N/A')}")

    # Build model
    model = build_model(cfg, task_type)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()

    # Load data
    dataloaders = create_dataloaders(cfg, pkl_path=pkl_path)

    results = {}
    for split, loader in dataloaders.items():
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                text   = batch['text'].to(device)
                audio  = batch['audio'].to(device)
                vision = batch['vision'].to(device)
                labels = batch['label'].to(device)
                al = batch.get('audio_len')
                vl = batch.get('vision_len')
                preds = model(
                    text=text, audio=audio, vision=vision,
                    audio_lengths=al.to(device) if al is not None else None,
                    vision_lengths=vl.to(device) if vl is not None else None,
                )
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        if task_type == 'emotion':
            metrics = compute_emotion_metrics(y_true, y_pred)
        else:
            metrics = compute_metrics(y_true, y_pred)

        results[split] = metrics

        if task_type == 'emotion':
            print(f"  [{split.upper()}] Mean_F1={metrics.get('mean_f1', 0):.4f}  "
                  f"Mean_Acc={metrics.get('mean_acc', 0):.4f}  "
                  f"Mean_MAE={metrics.get('mean_mae', 0):.4f}")
        else:
            print(f"  [{split.upper()}] MAE={metrics.get('mae', 0):.4f}  "
                  f"Corr={metrics.get('corr', 0):.4f}  "
                  f"Acc2={metrics.get('acc2', 0):.4f}")

    return results


# ============================================================
# Run all evaluations
# ============================================================
CKPT_DIR = '/content/checkpoints/phase1'
PKL_ALIGN   = '/content/data/MSA-Dataset/aligned_50.pkl'
PKL_UNALIGN = '/content/data/MSA-Dataset/unaligned_50.pkl'

all_models = [
    # Sentiment
    {'ckpt': f'{CKPT_DIR}/best_model_mult.pt',
     'label': 'MulT (aligned)',      'task': 'sentiment', 'model': 'mult',          'pkl': PKL_ALIGN},
    {'ckpt': f'{CKPT_DIR}/best_model_mult_unaligned.pt',
     'label': 'MulT (unaligned)',    'task': 'sentiment', 'model': 'mult',          'pkl': PKL_UNALIGN},
    {'ckpt': f'{CKPT_DIR}/best_model_improved_lstm.pt',
     'label': 'Improved LSTM',        'task': 'sentiment', 'model': 'improved_lstm', 'pkl': PKL_ALIGN},
    # Emotion
    {'ckpt': f'{CKPT_DIR}/best_model_mult_emotion.pt',
     'label': 'MulT Emotion (P0)',   'task': 'emotion',   'model': 'mult',          'pkl': PKL_ALIGN},
    # NOTE: best_model_mult_emotion_p1_focal.pt DIVERGED — skipped
]

evaluations = []
for m in all_models:
    try:
        results = evaluate_checkpoint(
            ckpt_path=m['ckpt'], task_type=m['task'],
            model_type=m['model'], pkl_path=m['pkl'], label=m['label'],
        )
        evaluations.append({'label': m['label'], 'task': m['task'], 'results': results})
    except FileNotFoundError:
        print(f'\nSKIPPED (not found): {m["ckpt"]}')
    except Exception as e:
        print(f'\nERROR on {m["label"]}: {e}')
        import traceback; traceback.print_exc()

print(f"\nDone. {len(evaluations)}/{len(all_models)} models evaluated.")

In [ ]:
# ============================================================
# CELL 6 — COMPARISON TABLE + SAVE
# ============================================================

print('\n' + '='*80)
print('FULL COMPARISON TABLE')
print('='*80)

# --- Sentiment ---
sent_rows = []
for ev in evaluations:
    if ev['task'] != 'sentiment': continue
    for split in ['valid', 'test']:
        r = ev['results'].get(split, {})
        sent_rows.append({
            'Model': ev['label'], 'Split': split,
            'MAE':   round(r.get('mae', 0), 4),
            'Corr':  round(r.get('corr', 0), 4),
            'Acc2':  round(r.get('acc2', 0), 4),
            'Acc5':  round(r.get('acc5', 0), 4),
            'Acc7':  round(r.get('acc7', 0), 4),
        })

sent_df = pd.DataFrame(sent_rows)
if len(sent_df):
    print('\n[SENTIMENT — Regression]')
    print(sent_df.to_string(index=False))

# --- Emotion ---
emo_rows = []
for ev in evaluations:
    if ev['task'] != 'emotion': continue
    for split in ['valid', 'test']:
        r = ev['results'].get(split, {})
        emo_rows.append({
            'Model': ev['label'], 'Split': split,
            'Mean_F1': round(r.get('mean_f1', 0), 4),
            'Mean_Acc': round(r.get('mean_acc', 0), 4),
            'Mean_MAE': round(r.get('mean_mae', 0), 4),
            'Happy_F1': round(r.get('happy_f1', 0), 4),
            'Sad_F1':   round(r.get('sad_f1', 0), 4),
            'Angry_F1': round(r.get('angry_f1', 0), 4),
            'Disgust_F1': round(r.get('disgust_f1', 0), 4),
            'Surprise_F1': round(r.get('surprise_f1', 0), 4),
            'Fear_F1':  round(r.get('fear_f1', 0), 4),
        })

emo_df = pd.DataFrame(emo_rows)
if len(emo_df):
    print('\n[EMOTION — Multi-label Classification]')
    print(emo_df.to_string(index=False))

# --- Save results ---
out_dir = '/content/drive/MyDrive/BCDA/outputs/phase1'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'all_evaluations.json')

output = {
    'sentiment': sent_df.to_dict('records') if len(sent_df) else [],
    'emotion':   emo_df.to_dict('records') if len(emo_df) else [],
    'metadata': {
        'device': str(device),
        'num_models_evaluated': len(evaluations),
    }
}

with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nResults saved: {out_path}')
print('\n' + '='*80)
print('DONE')
print('='*80)